In [1]:
import os
import numpy as np
import pandas as pd
import SimpleITK as sitk
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
import random
import json
import datetime

In [2]:
annotations = pd.read_csv('annotations.csv')
print("Total annotations:", len(annotations))

Total annotations: 1186


In [3]:
def world_to_voxel(world_coord, origin, spacing):
    return (world_coord - origin) / spacing

def extract_patch(image_array, center_voxel, patch_size=32):
    z, y, x = center_voxel[2], center_voxel[1], center_voxel[0]
    half = patch_size // 2
    z_min, z_max = max(0, z-half), min(image_array.shape[0], z+half)
    y_min, y_max = max(0, y-half), min(image_array.shape[1], y+half)
    x_min, x_max = max(0, x-half), min(image_array.shape[2], x+half)
    return image_array[z_min:z_max, y_min:y_max, x_min:x_max]

In [4]:
base_folders = ['subset0', 'subset1', 'subset2', 'subset3', 'subset4']
subset_paths = {}

for folder in base_folders:
    nested_path = os.path.join(folder, folder)
    direct_path = folder

    if os.path.exists(nested_path):
        actual_path = nested_path
    elif os.path.exists(direct_path):
        actual_path = direct_path
    else:
        print(f"{folder}: NOT FOUND")
        continue

    files = os.listdir(actual_path)
    mhd_count = len([f for f in files if f.endswith('.mhd')])
    subset_paths[folder] = actual_path
    print(f"{folder}: path='{actual_path}' | {mhd_count} scans found")

subset0: path='subset0\subset0' | 35 scans found
subset1: path='subset1' | 89 scans found
subset2: path='subset2' | 89 scans found
subset3: path='subset3' | 89 scans found
subset4: path='subset4' | 89 scans found


In [8]:
patch_size = 32
positive_patches = []
negative_patches = []
skipped_files = []

for folder, subset_path in subset_paths.items():
    mhd_files_subset = [f for f in os.listdir(subset_path) if f.endswith('.mhd')]
    print(f"\nProcessing {folder}: {len(mhd_files_subset)} scans")

    for mhd_file in mhd_files_subset:
        series_uid = mhd_file.replace('.mhd', '')
        scan_nodules = annotations[annotations['seriesuid'] == series_uid]

        if len(scan_nodules) == 0:
            continue

        try:
            img = sitk.ReadImage(os.path.join(subset_path, mhd_file))
            img_array = sitk.GetArrayFromImage(img)
            img_origin = np.array(img.GetOrigin())
            img_spacing = np.array(img.GetSpacing())
        except Exception as e:
            skipped_files.append(mhd_file)
            continue

        nodule_voxel_coords = []
        for idx, row in scan_nodules.iterrows():
            world_coord = np.array([row['coordX'], row['coordY'], row['coordZ']])
            voxel_coord = world_to_voxel(world_coord, img_origin, img_spacing).astype(int)
            nodule_voxel_coords.append(voxel_coord)

            patch = extract_patch(img_array, voxel_coord, patch_size)
            if patch.shape == (patch_size, patch_size, patch_size):
                positive_patches.append(patch)

        num_negatives_needed = len(scan_nodules)
        attempts = 0
        added = 0
        while added < num_negatives_needed and attempts < 50:
            attempts += 1
            rz = random.randint(patch_size, img_array.shape[0]-patch_size)
            ry = random.randint(patch_size, img_array.shape[1]-patch_size)
            rx = random.randint(patch_size, img_array.shape[2]-patch_size)
            random_voxel = np.array([rx, ry, rz])

            too_close = False
            for nv in nodule_voxel_coords:
                if np.linalg.norm(random_voxel - nv) < patch_size:
                    too_close = True
                    break

            if not too_close:
                patch = extract_patch(img_array, random_voxel, patch_size)
                if patch.shape == (patch_size, patch_size, patch_size):
                    negative_patches.append(patch)
                    added += 1

print(f"\n=== FINAL TOTALS ===")
print(f"Total positive patches (nodules): {len(positive_patches)}")
print(f"Total negative patches (normal tissue): {len(negative_patches)}")
print(f"Skipped files: {len(skipped_files)}")


Processing subset0: 35 scans

Processing subset1: 89 scans

Processing subset2: 89 scans

Processing subset3: 89 scans

Processing subset4: 89 scans

=== FINAL TOTALS ===
Total positive patches (nodules): 522
Total negative patches (normal tissue): 542
Skipped files: 1


In [9]:
X_positive = np.array(positive_patches, dtype=np.float32)
X_negative = np.array(negative_patches, dtype=np.float32)

y_positive = np.ones(len(X_positive))
y_negative = np.zeros(len(X_negative))

X = np.concatenate([X_positive, X_negative], axis=0)
y = np.concatenate([y_positive, y_negative], axis=0)

X = np.clip(X, -1000, 400)
X = (X + 1000) / 1400.0

indices = np.arange(len(X))
np.random.shuffle(indices)
X = X[indices]
y = y[indices]

print("Total samples:", len(X))
print("X shape:", X.shape)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Train samples:", len(X_train))
print("Test samples:", len(X_test))

Total samples: 1064
X shape: (1064, 32, 32, 32)
Train samples: 851
Test samples: 213


In [10]:
class SimpleNoduleClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv3d(1, 8, kernel_size=3, padding=1)
        self.pool = nn.MaxPool3d(2)
        self.conv2 = nn.Conv3d(8, 16, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(16 * 8 * 8 * 8, 32)
        self.fc2 = nn.Linear(32, 2)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = torch.relu(self.fc1(x))
        return self.fc2(x)

model = SimpleNoduleClassifier()
print(model)

SimpleNoduleClassifier(
  (conv1): Conv3d(1, 8, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
  (pool): MaxPool3d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv3d(8, 16, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
  (fc1): Linear(in_features=8192, out_features=32, bias=True)
  (fc2): Linear(in_features=32, out_features=2, bias=True)
)


In [11]:
X_train_t = torch.tensor(X_train, dtype=torch.float32).unsqueeze(1)
y_train_t = torch.tensor(y_train, dtype=torch.long)
X_test_t = torch.tensor(X_test, dtype=torch.float32).unsqueeze(1)
y_test_t = torch.tensor(y_test, dtype=torch.long)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epochs = 30
batch_size = 16

for epoch in range(num_epochs):
    model.train()
    permutation = torch.randperm(X_train_t.size(0))
    epoch_loss = 0
    correct = 0

    for i in range(0, X_train_t.size(0), batch_size):
        indices = permutation[i:i+batch_size]
        batch_x, batch_y = X_train_t[indices], y_train_t[indices]

        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == batch_y).sum().item()

    train_acc = correct / X_train_t.size(0)

    if (epoch+1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1}/{num_epochs} | Loss: {epoch_loss:.4f} | Train Accuracy: {train_acc*100:.1f}%")

print("\nTraining complete!")

Epoch 1/30 | Loss: 27.7658 | Train Accuracy: 73.1%
Epoch 5/30 | Loss: 6.8868 | Train Accuracy: 95.5%
Epoch 10/30 | Loss: 1.6549 | Train Accuracy: 98.9%
Epoch 15/30 | Loss: 0.1013 | Train Accuracy: 100.0%
Epoch 20/30 | Loss: 0.0260 | Train Accuracy: 100.0%
Epoch 25/30 | Loss: 0.0150 | Train Accuracy: 100.0%
Epoch 30/30 | Loss: 0.0095 | Train Accuracy: 100.0%

Training complete!


In [12]:
model.eval()
with torch.no_grad():
    test_outputs = model(X_test_t)
    _, test_predicted = torch.max(test_outputs, 1)
    test_acc = (test_predicted == y_test_t).float().mean().item()

print(f"Test Accuracy (unseen data): {test_acc*100:.2f}%")
print(f"Test samples: {len(y_test_t)}")

correct_nodule = ((test_predicted == 1) & (y_test_t == 1)).sum().item()
correct_normal = ((test_predicted == 0) & (y_test_t == 0)).sum().item()
false_positive = ((test_predicted == 1) & (y_test_t == 0)).sum().item()
false_negative = ((test_predicted == 0) & (y_test_t == 1)).sum().item()

print(f"\nCorrectly identified nodules: {correct_nodule}")
print(f"Correctly identified normal: {correct_normal}")
print(f"False positives: {false_positive}")
print(f"False negatives: {false_negative}")

Test Accuracy (unseen data): 94.84%
Test samples: 213

Correctly identified nodules: 115
Correctly identified normal: 87
False positives: 7
False negatives: 4


In [13]:
import shutil

os.makedirs('models', exist_ok=True)
os.makedirs('models/v1', exist_ok=True)

model_path = 'models/v1/model.pth'
torch.save(model.state_dict(), model_path)

metadata = {
    "version": "v1",
    "created_at": datetime.datetime.now().isoformat(),
    "training_dataset": "LUNA16 subset0-4",
    "total_scans_used": 391,
    "total_training_samples": len(X_train),
    "total_test_samples": len(X_test),
    "test_accuracy": round(test_acc * 100, 2),
    "correctly_identified_nodules": correct_nodule,
    "correctly_identified_normal": correct_normal,
    "false_positives": false_positive,
    "false_negatives": false_negative,
    "architecture": "SimpleNoduleClassifier (3D CNN)",
    "patch_size": 32,
    "epochs_trained": num_epochs,
    "status": "active"  
}

metadata_path = 'models/v1/metadata.json'
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print("Model saved to:", model_path)
print("Metadata saved to:", metadata_path)
print("\nMetadata:")
print(json.dumps(metadata, indent=2))

Model saved to: models/v1/model.pth
Metadata saved to: models/v1/metadata.json

Metadata:
{
  "version": "v1",
  "created_at": "2026-09-01T13:08:25.546674",
  "training_dataset": "LUNA16 subset0-4",
  "total_scans_used": 391,
  "total_training_samples": 851,
  "total_test_samples": 213,
  "test_accuracy": 94.84,
  "correctly_identified_nodules": 115,
  "correctly_identified_normal": 87,
  "false_positives": 7,
  "false_negatives": 4,
  "architecture": "SimpleNoduleClassifier (3D CNN)",
  "patch_size": 32,
  "epochs_trained": 30,
  "status": "active"
}
